# Week 5 — Evaluation Notebook

This notebook evaluates the wallet trust scoring system built in Weeks 1–4.

## Inputs
- `data/scoring_output.csv`
- `data/feature_table.csv` (optional, for feature-vs-score analysis)

## Goals
- Review score distribution
- Review trust-tier breakdown
- Inspect highest- and lowest-scoring wallets
- Analyze how key features relate to score
- Create simple simulated bot-like and human-like wallets for sanity checks


## 0) Imports + Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

scored = pd.read_csv('data/scoring_output.csv')

# feature_table.csv is optional but recommended for deeper analysis
try:
    features = pd.read_csv('data/feature_table.csv')
except FileNotFoundError:
    features = None

scored.head()

## 1) Basic Dataset Review

In [ ]:
print('Rows:', len(scored))
print('Columns:', scored.columns.tolist())
scored.describe(include='all')

## 2) Human Score Distribution

In [ ]:
scored['human_score'].describe()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(scored['human_score'], bins=15)
plt.title('Distribution of Human Trust Scores')
plt.xlabel('Human Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 3) Human Likelihood and Trust Tier Breakdown

In [ ]:
scored['human_likelihood'].value_counts(dropna=False)

In [ ]:
scored['trust_tier'].value_counts(dropna=False)

In [ ]:
plt.figure(figsize=(7, 5))
scored['trust_tier'].value_counts().plot(kind='bar')
plt.title('Trust Tier Distribution')
plt.xlabel('Trust Tier')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4) Highest- and Lowest-Scoring Wallets

In [ ]:
top_wallets = scored.sort_values('human_score', ascending=False).head(5)
bottom_wallets = scored.sort_values('human_score', ascending=True).head(5)

print('Top 5 wallets')
top_wallets[['wallet', 'human_score', 'human_likelihood', 'trust_tier', 'score_reasons']]

In [ ]:
print('Bottom 5 wallets')
bottom_wallets[['wallet', 'human_score', 'human_likelihood', 'trust_tier', 'score_reasons']]

## 5) Merge Scores with Features (Optional but Recommended)

In [ ]:
if features is not None:
    eval_df = features.merge(scored, on='wallet', how='left')
    eval_df.head()
else:
    print('feature_table.csv not found. Skip this section or add the file.')

## 6) Feature vs Score Analysis

In [ ]:
if features is not None:
    plt.figure(figsize=(7, 5))
    plt.scatter(eval_df['wallet_age_days'], eval_df['human_score'])
    plt.title('Wallet Age vs Human Score')
    plt.xlabel('Wallet Age (Days)')
    plt.ylabel('Human Score')
    plt.tight_layout()
    plt.show()
else:
    print('feature_table.csv not found. Skip this chart.')

In [ ]:
if features is not None:
    plt.figure(figsize=(7, 5))
    plt.scatter(eval_df['burst_tx_ratio'], eval_df['human_score'])
    plt.title('Burst Transaction Ratio vs Human Score')
    plt.xlabel('Burst Transaction Ratio')
    plt.ylabel('Human Score')
    plt.tight_layout()
    plt.show()
else:
    print('feature_table.csv not found. Skip this chart.')

In [ ]:
if features is not None:
    plt.figure(figsize=(7, 5))
    plt.scatter(eval_df['unique_counterparties'], eval_df['human_score'])
    plt.title('Unique Counterparties vs Human Score')
    plt.xlabel('Unique Counterparties')
    plt.ylabel('Human Score')
    plt.tight_layout()
    plt.show()
else:
    print('feature_table.csv not found. Skip this chart.')

## 7) Quick Correlation Check (Optional)

In [ ]:
if features is not None:
    numeric_cols = [
        'wallet_age_days', 'tx_count', 'tx_frequency', 'active_days',
        'avg_tx_per_active_day', 'mean_time_gap_hours', 'std_time_gap_hours',
        'burst_tx_ratio', 'unique_counterparties', 'interaction_entropy',
        'avg_tx_value_eth', 'total_value_eth', 'human_score'
    ]
    corr = eval_df[numeric_cols].corr(numeric_only=True)['human_score'].sort_values(ascending=False)
    corr
else:
    print('feature_table.csv not found. Skip this section.')

## 8) Simulated Sanity Check Wallets

This section creates a small simulated set of wallets to check whether the scoring logic behaves reasonably.


In [ ]:
def score_wallet_formula(row):
    score = 50

    if row['wallet_age_days'] >= 365:
        score += 15
    elif row['wallet_age_days'] >= 90:
        score += 8
    elif row['wallet_age_days'] < 30:
        score -= 12

    if row['active_days'] >= 30:
        score += 10
    elif row['active_days'] < 3:
        score -= 10

    if row['tx_frequency'] > 5:
        score -= 15
    elif row['tx_frequency'] > 1:
        score -= 5
    elif 0.01 <= row['tx_frequency'] <= 1:
        score += 5

    if row['burst_tx_ratio'] > 0.50:
        score -= 20
    elif row['burst_tx_ratio'] > 0.20:
        score -= 10
    else:
        score += 5

    if row['unique_counterparties'] >= 10:
        score += 10
    elif row['unique_counterparties'] <= 2:
        score -= 10

    if row['interaction_entropy'] >= 2:
        score += 8
    elif row['interaction_entropy'] < 0.5:
        score -= 8

    if row['mean_time_gap_hours'] > 0 and row['mean_time_gap_hours'] < 0.1:
        score -= 10
    elif row['mean_time_gap_hours'] >= 1:
        score += 5

    if row['std_time_gap_hours'] == 0:
        score -= 5
    elif row['std_time_gap_hours'] > 1:
        score += 3

    if 'unique_token_symbols' in row.index:
        if row['unique_token_symbols'] >= 5:
            score += 5
        elif row['unique_token_symbols'] == 0:
            score -= 2

    return max(0, min(100, score))

simulated = pd.DataFrame([
    {
        'profile': 'Human-like wallet',
        'wallet_age_days': 900,
        'active_days': 120,
        'tx_frequency': 0.6,
        'burst_tx_ratio': 0.03,
        'unique_counterparties': 25,
        'interaction_entropy': 2.7,
        'mean_time_gap_hours': 12,
        'std_time_gap_hours': 18,
        'unique_token_symbols': 8
    },
    {
        'profile': 'Bot-like wallet',
        'wallet_age_days': 7,
        'active_days': 1,
        'tx_frequency': 25,
        'burst_tx_ratio': 0.95,
        'unique_counterparties': 1,
        'interaction_entropy': 0.1,
        'mean_time_gap_hours': 0.01,
        'std_time_gap_hours': 0,
        'unique_token_symbols': 0
    }
])

simulated['simulated_score'] = simulated.apply(score_wallet_formula, axis=1)
simulated

## 9) Draft Evaluation Notes

In [ ]:
notes = []
notes.append(f"Average human score: {scored['human_score'].mean():.2f}")
notes.append(f"Highest human score: {scored['human_score'].max():.2f}")
notes.append(f"Lowest human score: {scored['human_score'].min():.2f}")
notes.append('High-scoring wallets generally show older age, broader activity spread, and lower burst behavior.')
notes.append('Low-scoring wallets generally show newer age, concentrated interactions, and higher burst behavior.')
notes.append('Main limitation: no labeled ground-truth dataset yet for formal validation.')

for n in notes:
    print('-', n)

## 10) Next Steps

- increase the number of wallets evaluated
- refine score weights and thresholds
- collect labeled bot/human examples
- build a dashboard or API layer for score inspection
